In [ ]:

# ==========================================
# TARGET DATA SETUP
# ==========================================
# Using the mapped target requirements
t_valid = np.array([-40.0, -23.5, -7.0, 9.5, 26.0, 42.5, 59.0, 75.5, 92.0, 108.5, 125.0])

# Using the interior points for regression
t_reg = np.array([-7.0, 9.5, 26.0, 42.5, 59.0, 75.5, 92.0, 108.5])
v_reg = np.array([0.900, 0.864, 0.810, 0.780, 0.751, 0.722, 0.679, 0.610])

# Identify saturated (excluded) for plotting consistency
exclude_temps = [-40.0, -23.5, 125.0]
is_excluded_mask = np.isin(t_valid, exclude_temps)
t_excluded = t_valid[is_excluded_mask]

# Using the *Ideal Vdc* values from the table (before actual hardware limits are plotted)
v_valid = np.array([0.900, 0.900, 0.900, 0.864, 0.810, 0.780, 0.751, 0.722, 0.679, 0.610, 0.610])
v_excluded = v_valid[is_excluded_mask]

# ==========================================
# LOAD GENERATED CTAT VOLTAGE DATA (CORNERS)
# ==========================================
def parse_voltage(val):
    if pd.isna(val): return np.nan
    val_str = str(val).strip()
    if val_str.endswith('m'):
        return float(val_str[:-1]) / 1000.0  # Convert mV to V
    return float(val_str)

file_path = 'VTEMP_CORNERS_NEWVARS_SLOPER_PrimeSim_default_final_Measurements_history_1_20260612_17_56_54.20.csv'

try:
    # Read the updated CSV format
    df_raw = pd.read_csv(file_path)
    
    # Process units for Vtemp
    df_raw['Vtemp_V'] = df_raw['Vtemp:dc_op'].apply(parse_voltage)
    df_raw['Temp'] = df_raw['Corner:Variable:temp'].astype(float)
    
    # Extract TT, SS, and FF corner dataframes based on 'Corner' column naming
    tt_data = df_raw[df_raw['Corner'].str.startswith('TT.')].sort_values('Temp')
    ss_data = df_raw[df_raw['Corner'].str.startswith('SS.')].sort_values('Temp')
    ff_data = df_raw[df_raw['Corner'].str.startswith('FF.')].sort_values('Temp')
    
    # Create the unified df_corners for plotting
    df_corners = pd.DataFrame({'XVAL': tt_data['Temp'].values})
    
    if not tt_data.empty:
        df_corners['TT'] = tt_data['Vtemp_V'].values
    if not ss_data.empty:
        df_corners['SS'] = ss_data['Vtemp_V'].values
    if not ff_data.empty:
        df_corners['FF'] = ff_data['Vtemp_V'].values

    # Set the primary circuit DataFrame using the TT values
    df_circuit = pd.DataFrame()
    if 'TT' in df_corners.columns:
        df_circuit['XVAL'] = df_corners['XVAL']
        df_circuit['Vtemp'] = df_corners['TT']
        df_circuit['Vtemp_mV'] = df_circuit['Vtemp'] * 1000

except FileNotFoundError:
    print(f"Error: '{file_path}' not found. Ensure the file is in the same directory.")
    df_corners = pd.DataFrame()
    df_circuit = pd.DataFrame()

# ==========================================
# 1. CALCULATE LINEAR REGRESSION TARGET
# ==========================================
slope, intercept = np.polyfit(t_reg, v_reg, 1)

v_fit_reg = slope * t_reg + intercept
ss_res = np.sum((v_reg - v_fit_reg)**2)
ss_tot = np.sum((v_reg - np.mean(v_reg))**2)
r_squared = 1 - (ss_res / ss_tot)

v_fit_plot = slope * t_valid + intercept

# ==========================================
# PLOTTING
# ==========================================

# ------------------------------------------
# Fig 1: ORIGINAL LINEAR REGRESSION 
# ------------------------------------------
fig1, ax1 = plt.subplots(figsize=(8, 5))

# Plotting in Volts
ax1.plot(t_reg, v_reg, marker='o', linestyle='', color='blue', label='Target Vdc Points (Used in Fit)')
ax1.plot(t_excluded, v_excluded, marker='x', markersize=8, linestyle='', color='orange', label='Saturated Points (Excluded)')

slope_mV = slope * 1000
intercept_mV = intercept * 1000

# Plotting the regression line in Volts
ax1.plot(t_valid, v_fit_plot, color='red', linewidth=2, label=f'Linear Fit: V(T) = {slope_mV:.3f} mV/°C * T + {intercept_mV:.1f} mV\n$R^2$ = {r_squared:.4f}')

ax1.set_title('Linear Regression: Required Compensation Voltage vs Temperature', fontweight='bold')
ax1.set_xlabel('Temperature (°C)')
ax1.set_ylabel('Required Vdc (V)')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(loc='best')
fig1.tight_layout()
plt.show()

# ------------------------------------------
# Fig 2: STANDARDIZED LINEAR REGRESSION
# ------------------------------------------
v_ref_25 = slope * 25.0 + intercept
v_reg_norm = v_reg / v_ref_25
v_excluded_norm = v_excluded / v_ref_25

slope_norm, intercept_norm = np.polyfit(t_reg, v_reg_norm, 1)
v_fit_plot_norm = slope_norm * t_valid + intercept_norm

fig2, ax2 = plt.subplots(figsize=(8, 5))

ax2.plot(t_reg, v_reg_norm, marker='o', linestyle='', color='purple', label='Standardized Vdc Points')
ax2.plot(t_excluded, v_excluded_norm, marker='x', markersize=8, linestyle='', color='orange', label='Saturated Points (Excluded)')

ax2.plot(t_valid, v_fit_plot_norm, color='green', linewidth=2, label=f'Standardized Fit: V_norm(T) = {slope_norm:.5f}*T + {intercept_norm:.5f}')

ax2.set_title('Standardized Tuning Curve (Normalized to Target Vdc @ 25°C)', fontweight='bold')
ax2.set_xlabel('Temperature (°C)')
ax2.set_ylabel('Vdc(T) / Vdc_target(25°C)')
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(loc='best')
fig2.tight_layout()
plt.show()

# ------------------------------------------
# Fig 3: ACTUAL CIRCUIT VS TARGET (Typical Corner Only)
# ------------------------------------------
if not df_circuit.empty:
    fig3, ax3 = plt.subplots(figsize=(8, 5))

    # Plot ideal regression target in mV
    ax3.plot(t_valid, v_fit_plot * 1000, color='red', linewidth=2, linestyle='--', label='Ideal Target (Calculated Linear Fit)')

    # Plot generated circuit voltage from TT corner in mV
    ax3.plot(df_circuit['XVAL'], df_circuit['Vtemp_mV'], color='black', linewidth=3, label='Generated Circuit Vtemp (TT)')

    ax3.set_title('Comparison: Generated Circuit Voltage (TT) vs Target Regression', fontweight='bold')
    ax3.set_xlabel('Temperature (°C)')
    ax3.set_ylabel('Voltage (mV)')
    ax3.grid(True, linestyle='--', alpha=0.6)
    ax3.legend(loc='best')
    fig3.tight_layout()
    plt.show()

# ==========================================
# 3. PRINT EXTRACTED EQUATIONS
# ==========================================
print(f"\n--> Extracted Tuning Curve (Absolute): Vdc(T) = {slope_mV:.3f} mV/°C * T + {intercept_mV:.1f} mV (R^2 = {r_squared:.4f})")
print(f"--> Standardized Tuning Curve (Relative to 25°C): V_norm(T) = {slope_norm:.6f} * T + {intercept_norm:.6f}\n")